### Simple ANN for Regression

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


data = pd.DataFrame({
    'area':np.array([100, 150, 200, 250, 300, 350, 400, 450, 500, 550]),
    'price': np.array([20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, 110000])
})

X = data['area'].values
y = data['price'].values


w1 = 1
b1 = 0
w2 = 1
b2 = 0

def relu(x):
    return np.maximum(0, x)

def mean_squared_error(y_true, y_pred):
    return (y_true - y_pred) ** 2


def relu_derivative(x):
    return np.where(x > 0, 1, 0)

def forward_pass(x: float):
    z1 = w1 * x + b1
    a1 = relu(z1)
    z2 = w2 * a1 + b2
    return z1, z2, a1

def back_propagation(x: float, y: float, y_pred: float, z1: float, a1: float, learning_rate=0.00001):
    global w1, b1, w2, b2
    l = mean_squared_error(y, y_pred)

    dl_db2 = 2 * (y - y_pred)
    dl_dw2 = dl_db2 * a1

    dl_db1 = dl_db2 * w2 * relu_derivative(z1)
    dl_dw1 = dl_db1 * x

    w2 += learning_rate * dl_dw2
    b2 += learning_rate * dl_db2
    w1 += learning_rate * dl_dw1
    b1 += learning_rate * dl_db1
    return b1, w1, b2, w2


epochs = 1000
for epoch in range(epochs):
    for i, data in enumerate(X):
        z1, y_pred, a1 = forward_pass(data)
        b1, w1, b2, w2 = back_propagation(data, y[i], y_pred, z1=z1, a1=a1)

# predict for a new area
new_area = 600
z1, predicted_price, _ = forward_pass(new_area)
print(f"Predicted price for area {new_area}: {predicted_price}")

Predicted price for area 600: 11778.815851536741


In [16]:
import numpy as np
import pandas as pd

# 1. Dataset - keeping X as a column vector (N, 1)
data = pd.DataFrame({
    'area': np.array([100, 150, 200, 250, 300, 350, 400, 450, 500, 550]),
    'price': np.array([20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, 110000])
})

# Reshape to (10, 1) so it's a formal matrix/vector
X = data['area'].values.reshape(-1, 1) 
y = data['price'].values.reshape(-1, 1)

# 2. Parameters (Scalars still work here because of NumPy broadcasting)
w1, b1 = 0.01, 0.0
w2, b2 = 0.01, 0.0
learning_rate = 0.0000001 

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# 3. Vectorized Forward Pass
def forward_pass(X_vec):
    # Every operation here now happens to all 10 rows simultaneously
    Z1 = X_vec * w1 + b1
    A1 = relu(Z1)
    Z2 = A1 * w2 + b2  # This is Y_pred (10, 1)
    return Z1, A1, Z2

# 4. Vectorized Backpropagation
def back_propagation(X_vec, Y_true, Y_pred, Z1, A1):
    global w1, b1, w2, b2
    
    m = X_vec.shape[0] # Number of samples (10)

    # 1. Output Layer Gradients
    # dL/dZ2 = 2 * (Y_pred - Y_true)
    dZ2 = 2 * (Y_pred - Y_true)
    
    # We average the gradients across all rows (divide by m)
    dw2 = np.sum(dZ2 * A1) / m
    db2 = np.sum(dZ2) / m

    # 2. Hidden Layer Gradients
    dA1 = dZ2 * w2
    dZ1 = dA1 * relu_derivative(Z1)
    
    dw1 = np.sum(dZ1 * X_vec) / m
    db1 = np.sum(dZ1) / m

    # 3. Update
    w2 -= learning_rate * dw2
    b2 -= learning_rate * db2
    w1 -= learning_rate * dw1
    b1 -= learning_rate * db1

# 5. The Training Loop (No inner loop!)
epochs = 1000
for epoch in range(epochs):
    Z1, A1, Y_pred = forward_pass(X)
    back_propagation(X, y, Y_pred, Z1, A1)

# 6. Predict
new_area = np.array([[600]])
_, _, predicted_price = forward_pass(new_area)
print(f"Predicted price for area 600: {predicted_price[0][0]:.2f}")

Predicted price for area 600: 12.95


## Simple ANN for House Price Prediction (Regression)

In [1]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    'area': np.array([100, 150, 200, 250, 300, 350]),
    'price': np.array([20000, 30000, 40000, 50000, 60000, 70000])
})



x = data['area'].values
y = data['price'].values

w1,b1,w2,b2 = 1,1,1,1


def relu(x: float):
    return np.maximum(0, x)

def derivative_relu(x: float):
    return np.where(x > 0, 1, 0)

def mean_squared_error(y_true: float, y_pred: float):
    return (y_true - y_pred) ** 2

def forward_pass(input_value: float,w1: float, b1: float, w2: float, b2: float):
    # first hidden layer
    z1: float = w1 * input_value + b1 
    a1 = relu(z1)

    # output layer
    z2 = w2 * a1 + b2

    y_hat = z2

    return z1, a1, z2, y_hat



def back_propagation(y_hat: float, y_true: float, z1: float, a1: float,input_value: float):
    global w1, b1, w2, b2
    # calculate the loss
    learning_rate = 0.01
    l = mean_squared_error(y_true, y_hat)
    # output layer gradients
    dl_dy_hat = 2 * (y_hat - y_true)
    dl_dw2 = dl_dy_hat * a1
    dl_db2 = dl_dy_hat

    # hidden layer gradients
    dl_da1 = dl_dy_hat * w2
    dl_dz1 = dl_da1 * derivative_relu(z1)
    dl_dw1 = dl_dz1 * input_value
    dl_db1 = dl_dz1

    # update weights and biases
    w2 -= learning_rate * dl_dw2
    b2 -= learning_rate * dl_db2
    w1 -= learning_rate * dl_dw1
    b1 -= learning_rate * dl_db1

    return w1, b1, w2, b2


epochs = 1000
for epoch in range(epochs):
    for i, data in enumerate(x):
        z1, a1, z2, y_hat = forward_pass(data, w1, b1, w2, b2)
        w1, b1, w2, b2 = back_propagation(y_hat, y[i], z1, a1, data)




# predict for a new area
new_area = 600
z1, a1, z2, predicted_price = forward_pass(new_area, w1, b1, w2, b2)
print(f"Predicted price for area {new_area}: {predicted_price}")

    


Predicted price for area 600: 45589.09737351722
